# LLM

> LLM-powered text rewriting, semantic grouping, and end-to-end pipeline.

In [ ]:
#| default_exp llm

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import json
import httpx
from typing import List, Dict, Any

from text_diff.diff import compute_diff
from text_diff.clean import clean_newlines, extract_newlines_to_blocks, add_accepted_flags
from text_diff.group import group_replacements, apply_semantic_groups

In [ ]:
#| export
DEFAULT_MODEL = "qwen/qwen3.5-flash-02-23"

## LLM rewrite

Ask an LLM to rewrite text according to a goal description.
Uses the OpenRouter API so any supported model can be swapped in.

In [ ]:
#| export
async def get_llm_rewrite(
    problem_description: str,
    current_text: str,
    api_key: str,
    model: str = "",
) -> str:
    """Ask an LLM to rewrite `current_text` to achieve `problem_description`.
    
    Returns the rewritten text string."""
    url = "https://openrouter.ai/api/v1/chat/completions"
    model = model or DEFAULT_MODEL

    system_instruction = (
        "You are a professional text editor.\n"
        "Your task: Rewrite the provided text to achieve the user's goal.\n"
        "IMPORTANT:\n"
        "- Return ONLY the rewritten text\n"
        "- Do NOT add explanations, commentary, or markdown formatting"
    )

    user_prompt = f"Goal: {problem_description}\n\nOriginal Text:\n{current_text}\n\nRewritten Text:"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.7,
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    async with httpx.AsyncClient() as client:
        response = await client.post(url, json=payload, headers=headers, timeout=60.0)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"].strip()

## Semantic grouping via LLM

After computing the diff, we can optionally ask an LLM to group
semantically related changes together. For example, if changing a
subject also requires changing the verb, those two replacements
should be presented as one logical edit.

In [ ]:
#| export
async def get_semantic_groups(
    diff_items: List[Dict[str, Any]],
    api_key: str,
    model: str = "",
) -> List[List[int]]:
    """Ask the LLM which diff items should be grouped together.
    
    Returns a list of groups, where each group is a list of indices to merge."""
    url = "https://openrouter.ai/api/v1/chat/completions"
    model = model or DEFAULT_MODEL

    system_instruction = (
        "You are a text diff analyzer. Group related changes together for better UX.\n\n"
        "GROUPING RULES:\n"
        "1. Group changes that are grammatically or semantically dependent\n"
        "2. Group consecutive changes that form a single logical edit\n"
        "3. Keep independent changes separate\n"
        "4. Never group changes separated by significant equal text (more than a few words)\n"
        '5. Never group changes across newline boundaries (type "equal" with text "\\n")\n\n'
        "Return ONLY a JSON array of groups, where each group is an array of indices to merge.\n"
        'Example: [[0, 1, 2], [4, 5]] means merge items 0,1,2 together and 4,5 together.\n'
        'If no grouping needed, return empty array: []'
    )

    user_prompt = (
        "Analyze this diff and suggest which consecutive items should be grouped together:\n\n"
        f"{json.dumps(diff_items, indent=2)}\n\n"
        "Return only the JSON array of groups (arrays of indices to merge)."
    )

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.3,
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    async with httpx.AsyncClient() as client:
        response = await client.post(url, json=payload, headers=headers, timeout=60.0)
        response.raise_for_status()
        data = response.json()
        content = data["choices"][0]["message"]["content"].strip()

        # Handle markdown code fences in response
        if "```" in content:
            start = content.find("[")
            end = content.rfind("]") + 1
            if start != -1 and end != 0:
                content = content[start:end]

        return json.loads(content)

In [ ]:
#| export
async def llm_semantic_grouping(
    ops: List[Dict[str, Any]],
    api_key: str,
    model: str = "",
) -> List[Dict[str, Any]]:
    """Use an LLM to intelligently group semantically related diff ops.
    
    Wraps `get_semantic_groups` + `apply_semantic_groups`.
    Falls back to returning `ops` unchanged if the LLM call fails."""
    if not ops or len(ops) <= 2:
        return ops
    if all(op.get("type") == "equal" for op in ops):
        return ops

    try:
        # Build simplified view for the LLM
        diff_for_llm = []
        for idx, op in enumerate(ops):
            op_type = op.get("type")
            if op_type == "equal":
                diff_for_llm.append({"index": idx, "type": "equal", "text": op.get("text", "")})
            elif op_type == "replace":
                diff_for_llm.append({"index": idx, "type": "replace", "old": op.get("old_text", ""), "new": op.get("new_text", "")})
            elif op_type == "add":
                diff_for_llm.append({"index": idx, "type": "add", "text": op.get("text", "")})
            elif op_type == "remove":
                diff_for_llm.append({"index": idx, "type": "remove", "text": op.get("text", "")})

        groups = await get_semantic_groups(diff_for_llm, api_key, model)
        return apply_semantic_groups(ops, groups)

    except Exception as e:
        print(f"LLM semantic grouping failed: {e}")
        return ops

## Diff pipeline

`process_diff` takes a raw original and rewritten text and runs the full
cleanup → grouping → newline extraction → flagging pipeline.

`request_text_improvement` is the top-level function: it asks the LLM
to rewrite, then diffs and processes the result.

In [ ]:
#| export
async def process_diff(
    original: str,
    rewritten: str,
    api_key: str = "",
    model: str = "",
    semantic_grouping: bool = True,
) -> List[Dict[str, Any]]:
    """Run the full diff pipeline: diff -> clean -> group -> extract newlines -> flag -> (optional) LLM grouping.
    
    Set `semantic_grouping=False` to skip the LLM semantic grouping step."""
    # Raw diff
    ops = compute_diff(original, rewritten)

    # Clean whitespace noise
    cleaned = clean_newlines(ops)
    if not cleaned:
        return [{"type": "equal", "text": original, "accepted": True}]

    # If no real changes remain, return early
    if all(op["type"] == "equal" for op in cleaned):
        return add_accepted_flags(cleaned)

    # Group consecutive replacements
    grouped = group_replacements(cleaned)

    # Extract newlines into their own blocks
    newline_extracted = extract_newlines_to_blocks(grouped)

    # Add accepted flags
    flagged = add_accepted_flags(newline_extracted)

    # Optional LLM semantic grouping
    if semantic_grouping and api_key:
        flagged = await llm_semantic_grouping(flagged, api_key, model)

    return flagged

In [ ]:
#| export
async def request_text_improvement(
    problem_description: str,
    current_text: str,
    api_key: str = "",
    model: str = "",
    semantic_grouping: bool = True,
) -> List[Dict[str, Any]]:
    """End-to-end: LLM rewrite then diff pipeline.
    
    1. Ask LLM to rewrite `current_text` according to `problem_description`
    2. Compute and process the diff between original and rewritten
    
    Returns a list of diff ops ready for display."""
    final_api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")

    if not final_api_key:
        return [
            {"text": "Error: API Key missing. ", "type": "equal", "accepted": True},
            {"text": "Please provide your OpenRouter API key.", "type": "add", "accepted": False},
        ]

    try:
        rewritten = await get_llm_rewrite(problem_description, current_text, final_api_key, model)
        return await process_diff(current_text, rewritten, final_api_key, model, semantic_grouping)
    except Exception as e:
        print(f"LLM Call Failed: {e}")
        return [
            {"text": "Error processing text: ", "type": "equal", "accepted": True},
            {"text": str(e), "type": "add", "accepted": False},
        ]

## Tests

The LLM functions require an API key so we can't run them in CI.
We test `process_diff` without semantic grouping to validate the pipeline,
and test `request_text_improvement` error handling.

In [ ]:
# process_diff without LLM grouping
result = await process_diff("the old cat", "the new cat", semantic_grouping=False)

# Should have equal + replace + equal, all with accepted flags
types = [op["type"] for op in result]
assert "replace" in types
assert all("accepted" in op for op in result)

# equal parts should be accepted, replace should not
for op in result:
    if op["type"] == "equal":
        assert op["accepted"] == True
    elif op["type"] == "replace":
        assert op["accepted"] == False

In [ ]:
# identical text -> all equal
result = await process_diff("same text", "same text", semantic_grouping=False)
assert all(op["type"] == "equal" for op in result)
assert all(op["accepted"] == True for op in result)

In [ ]:
# missing API key returns error ops (temporarily clear env var)
_saved_key = os.environ.pop("OPENROUTER_API_KEY", None)
result = await request_text_improvement("fix grammar", "hello world", api_key="")
if _saved_key: os.environ["OPENROUTER_API_KEY"] = _saved_key
assert any("API Key missing" in op.get("text", "") for op in result)

In [ ]:
# process_diff with multiline text
result = await process_diff("line one\nline two", "line one\nline three", semantic_grouping=False)
# Should contain a newline equal block
assert any(op.get("text") == "\n" and op["type"] == "equal" for op in result)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()